In [11]:
!pip install -q sentence-transformers

In [12]:
from google.colab import drive
drive.mount('/content/drive')

import os
import random
import numpy as np
import pandas as pd
import tensorflow as tf
from sentence_transformers import SentenceTransformer
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report, f1_score
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import LabelEncoder
from sklearn.ensemble import RandomForestClassifier

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [13]:
file_path = "/content/drive/MyDrive/AI_Builder/thai_gaming_toxic_dataset_v2_simple.csv"

df = pd.read_csv(file_path)

df.head()

,text,label
0,มึงโง่จน objective หายหมด,Toxic
1,ดีมาก ช่วยอีกทีมเต็มที่,Toxic
2,ไม่พร้อมก็อย่ากดมา ไม่ดูแมพเลย,Toxic
3,ถ้าจะกดสกิลมั่ว ควรหยุดก่อน,Toxic
4,เมื่อกี้ยังไม่ควรไล่ ค่อยๆเล่น,Normal


In [14]:
df = df.dropna(subset=["text", "label"]).copy()

df["text"] = df["text"].astype(str).str.strip()
df["label"] = df["label"].astype(str).str.strip()

texts = df["text"].tolist()
labels = df["label"].tolist()

print("Total rows:", len(df))
print(df["label"].value_counts())
print("Duplicates:", df.duplicated(subset=["text", "label"]).sum())
print("len(texts):", len(texts))
print("len(labels):", len(labels))

df.head()

Total rows: 1100
label
Toxic     550
Normal    550
Name: count, dtype: int64
Duplicates: 0
len(texts): 1100
len(labels): 1100


,text,label
0,มึงโง่จน objective หายหมด,Toxic
1,ดีมาก ช่วยอีกทีมเต็มที่,Toxic
2,ไม่พร้อมก็อย่ากดมา ไม่ดูแมพเลย,Toxic
3,ถ้าจะกดสกิลมั่ว ควรหยุดก่อน,Toxic
4,เมื่อกี้ยังไม่ควรไล่ ค่อยๆเล่น,Normal


In [15]:
X_train_text, X_test_text, y_train, y_test = train_test_split(
    texts,
    labels,
    test_size=0.3,
    random_state=42,
    stratify=labels
)

print("Train size:", len(X_train_text))
print("Test size:", len(X_test_text))

Train size: 770
Test size: 330


In [16]:
vectorizer = TfidfVectorizer(
    analyzer="char",
    ngram_range=(2, 5)
)

X_train = vectorizer.fit_transform(X_train_text)
X_test = vectorizer.transform(X_test_text)

print("X_train shape:", X_train.shape)
print("X_test shape:", X_test.shape)

X_train shape: (770, 7727)
X_test shape: (330, 7727)


In [17]:
unseen_path = "/content/drive/MyDrive/AI_Builder/thai_gaming_toxic_unseen_test_clean_244.csv"

unseen_df = pd.read_csv(unseen_path)

unseen_df = unseen_df.dropna(subset=["text", "label"]).copy()
unseen_df["text"] = unseen_df["text"].astype(str).str.strip()
unseen_df["label"] = unseen_df["label"].astype(str).str.strip()

unseen_texts = unseen_df["text"].tolist()
unseen_labels = unseen_df["label"].tolist()

print("Clean unseen rows:", len(unseen_df))
print(unseen_df["label"].value_counts())

main_text_set = set(df["text"].astype(str).str.strip())
unseen_text_set = set(unseen_df["text"].astype(str).str.strip())

overlap_texts = main_text_set.intersection(unseen_text_set)

print("Overlap with main dataset:", len(overlap_texts))

unseen_df.head()

Clean unseen rows: 244
label
Normal    122
Toxic     122
Name: count, dtype: int64
Overlap with main dataset: 0


,text,label
0,ไม่เป็นไรจริง ๆ แพ้ไฟต์เดียวไม่จบเกม,Normal
1,กูจะเปิดทางให้เดินออก,Normal
2,ไม่เป็นไร รอบหน้าเอาใหม่,Normal
3,ไอ้ทางเข้าป่ามี trap อยู่,Normal
4,ขอบคุณที่รอทีมก่อนเข้าไฟต์,Normal


In [18]:
toxic_keywords = [
    "กาก", "โง่", "ควาย", "ขยะ", "ปัญญาอ่อน",
    "แจก", "ภาระ", "ห่วย", "ตาย", "ทีมพัง",
    "เล่นไม่เป็น", "ถ่วงทีม", "ไม่มีสมอง"
]

def keyword_blacklist_predict(text):
    text = str(text)
    for word in toxic_keywords:
        if word in text:
            return "Toxic"
    return "Normal"

keyword_pred = [keyword_blacklist_predict(text) for text in unseen_texts]

tfidf_log_model = LogisticRegression(
    max_iter=2000,
    class_weight="balanced"
)

tfidf_log_model.fit(X_train, y_train)

tfidf_svm_model = LinearSVC(
    class_weight="balanced",
    max_iter=5000
)

tfidf_svm_model.fit(X_train, y_train)

emb_log_model = LogisticRegression(
    max_iter=2000,
    class_weight="balanced"
)

In [20]:
unseen_dataset_path = "/content/drive/MyDrive/AI_Builder/thai_gaming_toxic_unseen_test_clean_244.csv"

unseen_df = pd.read_csv(unseen_dataset_path)

unseen_df = unseen_df.dropna(subset=["text", "label"]).copy()
unseen_df["text"] = unseen_df["text"].astype(str).str.strip()
unseen_df["label"] = unseen_df["label"].astype(str).str.strip()

unseen_texts = unseen_df["text"].tolist()
unseen_labels = unseen_df["label"].tolist()

print("Total rows:", len(unseen_df))
print(unseen_df["label"].value_counts())


Total rows: 244
label
Normal    122
Toxic     122
Name: count, dtype: int64


In [24]:
embedder = SentenceTransformer(
    "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

In [25]:
X_train_emb = embedder.encode(
    X_train_text,
    show_progress_bar=True,
    convert_to_numpy=True
)

X_test_emb = embedder.encode(
    X_test_text,
    show_progress_bar=True,
    convert_to_numpy=True
)

X_unseen_emb = embedder.encode(
    unseen_texts,
    show_progress_bar=True,
    convert_to_numpy=True
)

print("X_train_emb shape:", X_train_emb.shape)
print("X_test_emb shape:", X_test_emb.shape)
print("X_unseen_emb shape:", X_unseen_emb.shape)

Batches:   0%|          | 0/25 [00:00<?, ?it/s]

Batches:   0%|          | 0/11 [00:00<?, ?it/s]

Batches:   0%|          | 0/8 [00:00<?, ?it/s]

X_train_emb shape: (770, 384)
X_test_emb shape: (330, 384)
X_unseen_emb shape: (244, 384)


In [28]:
tfidf_log_model = LogisticRegression(
    max_iter=2000,
    class_weight="balanced"
)

tfidf_log_model.fit(X_train, y_train)

tfidf_svm_model = LinearSVC(
    class_weight="balanced",
    max_iter=5000
)

tfidf_svm_model.fit(X_train, y_train)

emb_log_model = LogisticRegression(
    max_iter=2000,
    class_weight="balanced"
)

emb_log_model.fit(X_train_emb, y_train)

emb_svm_model = LinearSVC(
    class_weight="balanced",
    max_iter=5000
)

emb_svm_model.fit(X_train_emb, y_train)

emb_rf_model = RandomForestClassifier(
    n_estimators=200,
    random_state=42,
    class_weight="balanced"
)

emb_rf_model.fit(X_train_emb, y_train)

knn5_model = KNeighborsClassifier(
    n_neighbors=5,
    metric="cosine"
)

knn5_model.fit(X_train_emb, y_train)

knn9_model = KNeighborsClassifier(
    n_neighbors=9,
    metric="cosine"
)

knn9_model.fit(X_train_emb, y_train)

knn13_model = KNeighborsClassifier(
    n_neighbors=13,
    metric="cosine"
)

knn13_model.fit(X_train_emb, y_train)

X_unseen = vectorizer.transform(unseen_texts)

tfidf_log_pred = tfidf_log_model.predict(X_unseen)
tfidf_svm_pred = tfidf_svm_model.predict(X_unseen)

emb_log_pred = emb_log_model.predict(X_unseen_emb)
emb_svm_pred = emb_svm_model.predict(X_unseen_emb)
emb_rf_pred = emb_rf_model.predict(X_unseen_emb)

knn5_pred = knn5_model.predict(X_unseen_emb)
knn9_pred = knn9_model.predict(X_unseen_emb)
knn13_pred = knn13_model.predict(X_unseen_emb)

In [29]:
print("=== Method 1: Keyword / Blacklist basic baseline ===")
print("Accuracy:", accuracy_score(unseen_labels, keyword_pred))
print(classification_report(unseen_labels, keyword_pred))

print("=== Method 2: TF-IDF + Logistic Regression ===")
print("Accuracy:", accuracy_score(unseen_labels, tfidf_log_pred))
print(classification_report(unseen_labels, tfidf_log_pred))

print("=== Method 3: TF-IDF + Linear SVM ===")
print("Accuracy:", accuracy_score(unseen_labels, tfidf_svm_pred))
print(classification_report(unseen_labels, tfidf_svm_pred))

=== Method 1: Keyword / Blacklist basic baseline ===
Accuracy: 0.5983606557377049
              precision    recall  f1-score   support

      Normal       0.55      1.00      0.71       122
       Toxic       1.00      0.20      0.33       122

    accuracy                           0.60       244
   macro avg       0.78      0.60      0.52       244
weighted avg       0.78      0.60      0.52       244

=== Method 2: TF-IDF + Logistic Regression ===
Accuracy: 0.7868852459016393
              precision    recall  f1-score   support

      Normal       0.96      0.60      0.74       122
       Toxic       0.71      0.98      0.82       122

    accuracy                           0.79       244
   macro avg       0.83      0.79      0.78       244
weighted avg       0.83      0.79      0.78       244

=== Method 3: TF-IDF + Linear SVM ===
Accuracy: 0.8073770491803278
              precision    recall  f1-score   support

      Normal       0.95      0.65      0.77       122
       Toxic

In [30]:
print("=== Method 4: Sentence Embedding MiniLM + Logistic Regression ===")
print("Accuracy:", accuracy_score(unseen_labels, emb_log_pred))
print(classification_report(unseen_labels, emb_log_pred))

print("=== Method 5: Sentence Embedding MiniLM + Linear SVM ===")
print("Accuracy:", accuracy_score(unseen_labels, emb_svm_pred))
print(classification_report(unseen_labels, emb_svm_pred))

print("=== Method 6: Sentence Embedding MiniLM + Random Forest ===")
print("Accuracy:", accuracy_score(unseen_labels, emb_rf_pred))
print(classification_report(unseen_labels, emb_rf_pred))

=== Method 4: Sentence Embedding MiniLM + Logistic Regression ===
Accuracy: 0.819672131147541
              precision    recall  f1-score   support

      Normal       0.88      0.75      0.81       122
       Toxic       0.78      0.89      0.83       122

    accuracy                           0.82       244
   macro avg       0.83      0.82      0.82       244
weighted avg       0.83      0.82      0.82       244

=== Method 5: Sentence Embedding MiniLM + Linear SVM ===
Accuracy: 0.8278688524590164
              precision    recall  f1-score   support

      Normal       0.89      0.75      0.81       122
       Toxic       0.78      0.91      0.84       122

    accuracy                           0.83       244
   macro avg       0.84      0.83      0.83       244
weighted avg       0.84      0.83      0.83       244

=== Method 6: Sentence Embedding MiniLM + Random Forest ===
Accuracy: 0.819672131147541
              precision    recall  f1-score   support

      Normal       0.91

In [31]:
SEED = 21

os.environ["PYTHONHASHSEED"] = str(SEED)
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

tf.keras.backend.clear_session()

le = LabelEncoder()

y_train_num = le.fit_transform(y_train)

X_train_emb_float = np.asarray(X_train_emb).astype("float32")
X_unseen_emb_float = np.asarray(X_unseen_emb).astype("float32")

X_dense_train, X_dense_val, y_dense_train, y_dense_val = train_test_split(
    X_train_emb_float,
    y_train_num,
    test_size=0.2,
    random_state=SEED,
    stratify=y_train_num
)

dense_model = tf.keras.Sequential([
    tf.keras.layers.Input(shape=(X_train_emb_float.shape[1],)),

    tf.keras.layers.Dense(128, activation="relu"),
    tf.keras.layers.Dropout(0.3),

    tf.keras.layers.Dense(64, activation="relu"),
    tf.keras.layers.Dropout(0.2),

    tf.keras.layers.Dense(1, activation="sigmoid")
])

dense_model.compile(
    optimizer="adam",
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

history = dense_model.fit(
    X_dense_train,
    y_dense_train,
    validation_data=(X_dense_val, y_dense_val),
    epochs=15,
    batch_size=32,
    verbose=1
)

dense_prob = dense_model.predict(X_unseen_emb_float, verbose=0).ravel()

dense_pred_num = (dense_prob >= 0.5).astype(int)

dense_pred = le.inverse_transform(dense_pred_num)

dense_accuracy = accuracy_score(unseen_labels, dense_pred)
dense_macro_f1 = f1_score(unseen_labels, dense_pred, average="macro")



Epoch 1/15
20/20 ━━━━━━━━━━━━━━━━━━━━ 7s 53ms/step - accuracy: 0.7240 - loss: 0.5747 - val_accuracy: 0.8896 - val_loss: 0.3998
Epoch 2/15
20/20 ━━━━━━━━━━━━━━━━━━━━ 1s 35ms/step - accuracy: 0.9107 - loss: 0.2820 - val_accuracy: 0.9286 - val_loss: 0.2158
Epoch 3/15
20/20 ━━━━━━━━━━━━━━━━━━━━ 1s 32ms/step - accuracy: 0.9529 - loss: 0.1523 - val_accuracy: 0.9481 - val_loss: 0.1762
Epoch 4/15
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 18ms/step - accuracy: 0.9594 - loss: 0.1055 - val_accuracy: 0.9481 - val_loss: 0.1568
Epoch 5/15
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.9692 - loss: 0.0895 - val_accuracy: 0.9481 - val_loss: 0.1533
Epoch 6/15
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.9773 - loss: 0.0591 - val_accuracy: 0.9481 - val_loss: 0.1486
Epoch 7/15
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - accuracy: 0.9870 - loss: 0.0464 - val_accuracy: 0.9610 - val_loss: 0.1460
Epoch 8/15
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.9903 - loss: 0.0359 - val_accuracy: 0.9610 - val_l

In [32]:
print("=== Method 7: Dense Layer on Sentence Embedding MiniLM + Fixed Seed ===")
print("Accuracy:", accuracy_score(unseen_labels, dense_pred))
print(classification_report(unseen_labels, dense_pred))

=== Method 7: Dense Layer on Sentence Embedding MiniLM + Fixed Seed ===
Accuracy: 0.8278688524590164
              precision    recall  f1-score   support

      Normal       0.88      0.76      0.82       122
       Toxic       0.79      0.89      0.84       122

    accuracy                           0.83       244
   macro avg       0.83      0.83      0.83       244
weighted avg       0.83      0.83      0.83       244



In [33]:
print("=== Method 8: Sentence Embedding MiniLM + KNN5 ===")
print("Accuracy:", accuracy_score(unseen_labels, knn5_pred))
print(classification_report(unseen_labels, knn5_pred))

print("=== Method 9: Sentence Embedding MiniLM + KNN9 ===")
print("Accuracy:", accuracy_score(unseen_labels, knn9_pred))
print(classification_report(unseen_labels, knn9_pred))

print("=== Method 10: Sentence Embedding MiniLM + KNN13 ===")
print("Accuracy:", accuracy_score(unseen_labels, knn13_pred))
print(classification_report(unseen_labels, knn13_pred))

=== Method 8: Sentence Embedding MiniLM + KNN5 ===
Accuracy: 0.8442622950819673
              precision    recall  f1-score   support

      Normal       0.85      0.84      0.84       122
       Toxic       0.84      0.85      0.85       122

    accuracy                           0.84       244
   macro avg       0.84      0.84      0.84       244
weighted avg       0.84      0.84      0.84       244

=== Method 9: Sentence Embedding MiniLM + KNN9 ===
Accuracy: 0.8565573770491803
              precision    recall  f1-score   support

      Normal       0.85      0.86      0.86       122
       Toxic       0.86      0.85      0.86       122

    accuracy                           0.86       244
   macro avg       0.86      0.86      0.86       244
weighted avg       0.86      0.86      0.86       244

=== Method 10: Sentence Embedding MiniLM + KNN13 ===
Accuracy: 0.860655737704918
              precision    recall  f1-score   support

      Normal       0.86      0.86      0.86       

In [34]:
from sklearn.metrics import accuracy_score, f1_score
import pandas as pd

def get_result(model_name, y_pred):
    return {
        "method": model_name,
        "accuracy": accuracy_score(unseen_labels, y_pred),
        "accuracy_percent": round(accuracy_score(unseen_labels, y_pred) * 100, 2),
        "macro_f1": round(f1_score(unseen_labels, y_pred, average="macro"), 4)
    }

final_results_ordered_df = pd.DataFrame([
    get_result("Method 1: Keyword / Blacklist basic baseline", keyword_pred),
    get_result("Method 2: TF-IDF + Logistic Regression", tfidf_log_pred),
    get_result("Method 3: TF-IDF + Linear SVM", tfidf_svm_pred),
    get_result("Method 4: Sentence Embedding MiniLM + Logistic Regression", emb_log_pred),
    get_result("Method 5: Sentence Embedding MiniLM + Linear SVM", emb_svm_pred),
    get_result("Method 6: Sentence Embedding MiniLM + Random Forest", emb_rf_pred),
    get_result("Method 7: Dense Layer + Fixed Seed", dense_pred),
    get_result("Method 8: Sentence Embedding MiniLM + KNN5", knn5_pred),
    get_result("Method 9: Sentence Embedding MiniLM + KNN9", knn9_pred),
    get_result("Method 10: Sentence Embedding MiniLM + KNN13", knn13_pred)
])

final_results_ordered_df

,method,accuracy,accuracy_percent,macro_f1
0,Method 1: Keyword / Blacklist basic baseline,0.598361,59.84,0.5211
1,Method 2: TF-IDF + Logistic Regression,0.786885,78.69,0.7790
2,Method 3: TF-IDF + Linear SVM,0.807377,80.74,0.8023
3,Method 4: Sentence Embedding MiniLM + Logistic...,0.819672,81.97,0.8187
4,Method 5: Sentence Embedding MiniLM + Linear SVM,0.827869,82.79,0.8267
5,Method 6: Sentence Embedding MiniLM + Random F...,0.819672,81.97,0.8176
6,Method 7: Dense Layer + Fixed Seed,0.827869,82.79,0.8271
7,Method 8: Sentence Embedding MiniLM + KNN5,0.844262,84.43,0.8443
8,Method 9: Sentence Embedding MiniLM + KNN9,0.856557,85.66,0.8566
9,Method 10: Sentence Embedding MiniLM + KNN13,0.860656,86.07,0.8607
